# Fake News Detector

## Importing the dependencies

In [2]:
import numpy as np
import pandas as pd
import re
# stopwords that doesn't add value to a paragraph(where what, is all else)
from nltk.corpus import stopwords
import nltk
# gives root word for particular word
from nltk.stem.porter import PorterStemmer
# for converting text to feature vector
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
# 
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
# Stemming is the processing of taking the word and removing prefix and suffix of word and returns the root word.

In [3]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/ishukumar5663/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [4]:
# printing the stopwords
print(stopwords.words("english"))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

## Data Pre-processing

In [5]:
# Load dataset into pandas DataFrame
news_dataset = pd.read_csv("./train.csv")


In [6]:
news_dataset.shape

(20800, 5)

In [7]:
news_dataset.head()

,id,title,author,text,label
0,0,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus,House Dem Aide: We Didn’t Even See Comey’s Let...,1
1,1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Daniel J. Flynn,Ever get the feeling your life circles the rou...,0
2,2,Why the Truth Might Get You Fired,Consortiumnews.com,"Why the Truth Might Get You Fired October 29, ...",1
3,3,15 Civilians Killed In Single US Airstrike Hav...,Jessica Purkiss,Videos 15 Civilians Killed In Single US Airstr...,1
4,4,Iranian woman jailed for fictional unpublished...,Howard Portnoy,Print \nAn Iranian woman has been sentenced to...,1


In [8]:
#  counting the number of missing values in the dataset
news_dataset.isnull().sum()

id           0
title      558
author    1957
text        39
label        0
dtype: int64

In [9]:
# replacing the null values with empty string

news_dataset = news_dataset.fillna("")

In [10]:
# merging the author name, title and text
news_dataset['content'] = news_dataset['author'] + " " + news_dataset['title'] + ' ' + news_dataset["text"]

In [11]:
news_dataset

,id,title,author,text,label,content
0,0,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus,House Dem Aide: We Didn’t Even See Comey’s Let...,1,Darrell Lucus House Dem Aide: We Didn’t Even S...
1,1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Daniel J. Flynn,Ever get the feeling your life circles the rou...,0,"Daniel J. Flynn FLYNN: Hillary Clinton, Big Wo..."
2,2,Why the Truth Might Get You Fired,Consortiumnews.com,"Why the Truth Might Get You Fired October 29, ...",1,Consortiumnews.com Why the Truth Might Get You...
3,3,15 Civilians Killed In Single US Airstrike Hav...,Jessica Purkiss,Videos 15 Civilians Killed In Single US Airstr...,1,Jessica Purkiss 15 Civilians Killed In Single ...
4,4,Iranian woman jailed for fictional unpublished...,Howard Portnoy,Print \nAn Iranian woman has been sentenced to...,1,Howard Portnoy Iranian woman jailed for fictio...
...,...,...,...,...,...,...
20795,20795,Rapper T.I.: Trump a ’Poster Child For White S...,Jerome Hudson,Rapper T. I. unloaded on black celebrities who...,0,Jerome Hudson Rapper T.I.: Trump a ’Poster Chi...
20796,20796,"N.F.L. Playoffs: Schedule, Matchups and Odds -...",Benjamin Hoffman,When the Green Bay Packers lost to the Washing...,0,"Benjamin Hoffman N.F.L. Playoffs: Schedule, Ma..."
20797,20797,Macy’s Is Said to Receive Takeover Approach by...,Michael J. de la Merced and Rachel Abrams,The Macy’s of today grew from the union of sev...,0,Michael J. de la Merced and Rachel Abrams Macy...
20798,20798,"NATO, Russia To Hold Parallel Exercises In Bal...",Alex Ansary,"NATO, Russia To Hold Parallel Exercises In Bal...",1,"Alex Ansary NATO, Russia To Hold Parallel Exer..."


In [12]:
# seperating the data and label
X = news_dataset.drop(columns=["label", "id"], axis=0)
Y = news_dataset["label"]

In [13]:
print(X)
print(Y)

                                                   title  \
0      House Dem Aide: We Didn’t Even See Comey’s Let...   
1      FLYNN: Hillary Clinton, Big Woman on Campus - ...   
2                      Why the Truth Might Get You Fired   
3      15 Civilians Killed In Single US Airstrike Hav...   
4      Iranian woman jailed for fictional unpublished...   
...                                                  ...   
20795  Rapper T.I.: Trump a ’Poster Child For White S...   
20796  N.F.L. Playoffs: Schedule, Matchups and Odds -...   
20797  Macy’s Is Said to Receive Takeover Approach by...   
20798  NATO, Russia To Hold Parallel Exercises In Bal...   
20799                          What Keeps the F-35 Alive   

                                          author  \
0                                  Darrell Lucus   
1                                Daniel J. Flynn   
2                             Consortiumnews.com   
3                                Jessica Purkiss   
4                  

## Stemming

Stemming is the process of reducing a word to its root word

Ex: Actor, actress, acting --> act (stemming)

In [14]:
port_stem = PorterStemmer()

In [15]:
def stemming(content: str) -> str:
    # exclusion of a-z and A-Z, replace them with space
    stemmed_content = re.sub('[^a-zA-Z]',' ',content)
    stemmed_content = stemmed_content.lower()
    stemmed_content = stemmed_content.split()
    stemmed_content = [port_stem.stem(word) for word in stemmed_content if not word in stopwords.words("english")]
    #  just hover over join to understand
    stemmed_content = " ".join(stemmed_content)
    return stemmed_content

In [16]:

#! Need to run once 
news_dataset["content"] = news_dataset['content'].apply(stemming)
news_dataset.to_csv("./model_train.csv")

In [17]:
X.head()

,title,author,text,content
0,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus House Dem Aide: We Didn’t Even S...
1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Daniel J. Flynn,Ever get the feeling your life circles the rou...,"Daniel J. Flynn FLYNN: Hillary Clinton, Big Wo..."
2,Why the Truth Might Get You Fired,Consortiumnews.com,"Why the Truth Might Get You Fired October 29, ...",Consortiumnews.com Why the Truth Might Get You...
3,15 Civilians Killed In Single US Airstrike Hav...,Jessica Purkiss,Videos 15 Civilians Killed In Single US Airstr...,Jessica Purkiss 15 Civilians Killed In Single ...
4,Iranian woman jailed for fictional unpublished...,Howard Portnoy,Print \nAn Iranian woman has been sentenced to...,Howard Portnoy Iranian woman jailed for fictio...


In [18]:
# Reading from saved output
# X = pd.read_csv("./model_x.csv")



In [19]:
# seperating data and label
X = news_dataset["content"].values
Y = news_dataset["label"].values

In [20]:
X,Y

(<StringArray>
 [                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [21]:
Y.shape

(20800,)

In [22]:
vectorizer = TfidfVectorizer()
#? converting the textual data to numerical data, assigns a word power that a word appearing number of times and assign inverse power to it, idf is inverse so if a word appearing multiple times doesn't have much important
vectorizer.fit(X)

X = vectorizer.transform(X)

In [23]:
print(X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 5144938 stored elements and shape (20800, 111501)>
  Coords	Values
  (0, 325)	0.04351705682996119
  (0, 521)	0.022974569956367194
  (0, 635)	0.038727100898403345
  (0, 873)	0.01573563765182704
  (0, 922)	0.016392735476282342
  (0, 1298)	0.021078125206502117
  (0, 1610)	0.01797988302277085
  (0, 1886)	0.11545306338629945
  (0, 3022)	0.046620910610995464
  (0, 3042)	0.01902260509613337
  (0, 3377)	0.011797233080872101
  (0, 3743)	0.030879171097770226
  (0, 4156)	0.016679444655381123
  (0, 4260)	0.01942127157583213
  (0, 4306)	0.026424556938478543
  (0, 4588)	0.01769571497502257
  (0, 4772)	0.025939050637307268
  (0, 4822)	0.041683522324264644
  (0, 4839)	0.014401489020964033
  (0, 6421)	0.018224972680710137
  (0, 6964)	0.02064947765731587
  (0, 8556)	0.021643407923861314
  (0, 9072)	0.014237130761490214
  (0, 10846)	0.04758422449372864
  (0, 12251)	0.02777858540400552
  :	:
  (20799, 106830)	0.03658229119526847
  (20799, 10696

## Splitting from training to test data

In [24]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, stratify=Y ,random_state=2)

## Training the model

In [25]:
model = LogisticRegression()

In [26]:
model.fit(X_train, Y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

## Evaluation of model

In [27]:
X_train_prediction = model.predict(X_train)
training_data_accuracy = accuracy_score(X_train_prediction, Y_train)
print("Accuracy score on training data: ", training_data_accuracy)


Accuracy score on training data:  0.9785456730769231


In [28]:
X_test_pred = model.predict(X_test)
test_data_accuracy = accuracy_score(X_test_pred, Y_test)
print("Accuracy score on test data: ", test_data_accuracy)

Accuracy score on test data:  0.9526442307692308


## Making a predictive system

In [29]:
X_new = X_test[0]

prediction = model.predict(X_new)
#? 1 is fake and 0 is real
if(prediction[0] == 0):
    print("Real News")
else:
    print("Fake news")

print(Y_test[0])

Fake news
1


In [30]:
# Fake news
news = "Donald Trump Sends Out Embarrassing New Year’s Eve Message; This is Disturbing	Donald Trump just couldn t wish all Americans a Happy New Year and leave it at that. Instead, he had to give a shout out to his enemies, haters and  the very dishonest fake news media.  The former reality show star had just one job to do and he couldn t do it. As our Country rapidly grows stronger and smarter, I want to wish all of my friends, supporters, enemies, haters, and even the very dishonest Fake News Media, a Happy and Healthy New Year,  President Angry Pants tweeted.  2018 will be a great year for America! As our Country rapidly grows stronger and smarter, I want to wish all of my friends, supporters, enemies, haters, and even the very dishonest Fake News Media, a Happy and Healthy New Year. 2018 will be a great year for America!  Donald J. Trump (@realDonaldTrump) December 31, 2017Trump s tweet went down about as welll as you d expect.What kind of president sends a New Year s greeting like this despicable, petty, infantile gibberish? Only Trump! His lack of decency won t even allow him to rise above the gutter long enough to wish the American citizens a happy new year!  Bishop Talbert Swan (@TalbertSwan) December 31, 2017no one likes you  Calvin (@calvinstowell) December 31, 2017Your impeachment would make 2018 a great year for America, but I ll also accept regaining control of Congress.  Miranda Yaver (@mirandayaver) December 31, 2017Do you hear yourself talk? When you have to include that many people that hate you you have to wonder? Why do the they all hate me?  Alan Sandoval (@AlanSandoval13) December 31, 2017Who uses the word Haters in a New Years wish??  Marlene (@marlene399) December 31, 2017You can t just say happy new year?  Koren pollitt (@Korencarpenter) December 31, 2017Here s Trump s New Year s Eve tweet from 2016.Happy New Year to all, including to my many enemies and those who have fought me and lost so badly they just don t know what to do. Love!  Donald J. Trump (@realDonaldTrump) December 31, 2016This is nothing new for Trump. He s been doing this for years.Trump has directed messages to his  enemies  and  haters  for New Year s, Easter, Thanksgiving, and the anniversary of 9/11. pic.twitter.com/4FPAe2KypA  Daniel Dale (@ddale8) December 31, 2017Trump s holiday tweets are clearly not presidential.How long did he work at Hallmark before becoming President?  Steven Goodine (@SGoodine) December 31, 2017He s always been like this . . . the only difference is that in the last few years, his filter has been breaking down.  Roy Schulze (@thbthttt) December 31, 2017Who, apart from a teenager uses the term haters?  Wendy (@WendyWhistles) December 31, 2017he s a fucking 5 year old  Who Knows (@rainyday80) December 31, 2017So, to all the people who voted for this a hole thinking he would change once he got into power, you were wrong! 70-year-old men don t change and now he s a year older.Photo by Andrew Burton/Getty Images.	News	December 31, 2017"
news = stemming(news)
news_tdidf = vectorizer.transform([news])
pred = model.predict(news_tdidf)
print(pred)

[1]


In [33]:
# Riyal News
news = "Duterte berates Canada's Trudeau at end of Philippines summit	MANILA (Reuters) - Philippines President Rodrigo Duterte attacked Canada’s Justin Trudeau at the end of a summit of Asian and Western nations for raising questions about his war on drugs, a topic skirted by other leaders, including U.S. President Donald Trump. At the traditional news conference by the host nation at the end of the summit on Tuesday, Duterte was asked how he had responded to the Canadian prime minister raising the issue of human rights and extra-judicial killings in his anti-drugs drive. “I said I will not explain. It is a personal and official insult,” the Philippines president said in the course of a rambling answer, although he did not refer to Trudeau by name. “I only answer to the Filipino. I will not answer to any other bullshit, especially foreigners. Lay off.” Earlier in the day, Trudeau told a news conference that during his meeting with Duterte “the president was receptive to my comments and it was throughout a very cordial and positive exchange”. Human rights activists had been hoping that leaders at the summit, including Trump, would raise the issue of the thousands of users and small-time pushers killed in the campaign that was launched by Duterte after he took office in mid-2016. His government says the police act in self-defense during drug-busts, but critics say executions are taking place with no accountability. There was no pressure from Trump on the drugs war when he met Duterte on Monday and the U.S. president later said the two had a “great relationship”. A joint statement after the meeting only said the two sides “underscored that human rights and the dignity of human life are essential, and agreed to continue mainstreaming the human rights agenda in their national programs.” Duterte cursed Trump’s predecessor, Barack Obama, last year for raising concerns about the war on drugs and he subsequently declared that he was breaking ties with the United States, a close ally of the Philippines since World War Two. The relationship appears to have got back on track after the bonhomie between him and Trump. Trudeau also said that he raised the issue of the exodus of Rohingya during a meeting with Myanmar leader Aung San Suu Kyi, another sensitive topic bypassed by most other leaders, although he did not mention the Muslim minority by name. “This is a tremendous concern to Canada and to many, many countries around the world,” he said. The government in mostly-Buddhist Myanmar regards the Rohingya as illegal immigrants from Bangladesh and does not recognize the term. Over 600,000 Rohingya have fled to refugee camps in Bangladesh since military clearance operations were launched in response to attacks by Rohingya militants on Aug. 25.  The plight of the Rohingya has brought outrage from around the world and the United Nations has called the operations ethnic cleansing. There have been calls for democracy champion Suu Kyi to be stripped of the Nobel peace prize she won in 1991 because she has not condemned the military’s actions. Some countries in the 10-member Association of Southeast Asian Nations (ASEAN), particularly Muslim-majority Malaysia, have voiced strong concern over the issue recently. However, in keeping with ASEAN’s principle of non-interference in each others’ internal affairs, it appeared to have been put aside at the summit, which brought Southeast Asian nations together with the United States, Russia, Japan, China, India, Australia, New Zealand and Canada. Duterte reported that China had agreed at the summit to work on a code of conduct in the South China Sea with ASEAN nations to ease tensions over disputed claims to the busy and resource-rich waterway. The group also signed agreements on protecting migrant labor and fighting terrorism and cybercrime. Trump skipped the plenary session of the summit because of scheduling delays, but he said his marathon trip to Asia had been a “tremendous” success. He told reporters on Air Force One that he had delivered his prepared remarks during a lunch before the summit meeting.  Trump said at least $300 billion, possibly triple that figure, of deals had been agreed in the trip. He did not elaborate. “We’ve explained that the United States is open for trade but we want reciprocal, we want fair trade for the United States,” he said. Trade and concern about possible protectionism under Trump’s “America First” agenda have come up during his visit to the region, which included stops in Japan, South Korea, China, Vietnam before concluding in the Philippines. After Trump left Manila, a group of Asia-Pacific nations pursuing a separate Beijing-backed trade deal that does not include the United States agreed to “intensify efforts” in 2018 to bring their negotiations to a conclusion. The Regional Comprehensive Economic Partnership (RCEP) appeared to have been given new impetus at the summit by Trump’s withdrawal from the Trans-Pacific Partnership (TPP) trade agreement, to which China is not party. The two trade deals are not mutually exclusive. ASEAN is joined in the RCEP talks by China, India, Australia, New Zealand, Japan and South Korea. 	politicsNews	November 14, 2017"
news = stemming(news)
news_tdidf = vectorizer.transform([news])
pred = model.predict(news_tdidf)
print(pred)


[0]
